# Warmup Length Sweep Analysis (Point 2)

Loads `gap_warmup_sweep_results.json` and answers: how long does the warmup need to be?

- Fig 1: CD_fc vs training step for each warmup length (mean ± std)
- Fig 2: CD_fc at step 60 vs warmup length  ← the main summary
- Fig 3: Soft margin and accuracy at step 60 vs warmup length

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

with open('gap_warmup_sweep_results.json') as f:
    data = json.load(f)

results = data['results']
warmup_lengths = sorted(set(r['n_warmup_long'] for r in results))
print('Warmup lengths:', warmup_lengths)

In [ ]:
def get_metric_traj(results, warmup_len, metric_path):
    """metric_path: 'overlaps_fc.CD' or 'soft_margin_C' etc."""
    runs = [r for r in results if r['n_warmup_long'] == warmup_len]
    out = []
    for r in runs:
        traj = []
        for step in r['trajectory']:
            keys = metric_path.split('.')
            val = step
            for k in keys:
                val = val[k]
            traj.append(float(val))
        out.append(traj)
    return np.array(out)  # [n_seeds, n_steps]

cmap = plt.cm.viridis
colors = {wl: cmap(i / max(1, len(warmup_lengths) - 1))
          for i, wl in enumerate(warmup_lengths)}

In [ ]:
# Fig 1: CD_fc trajectory per warmup length
fig, ax = plt.subplots(figsize=(10, 5))
for wl in warmup_lengths:
    arr = get_metric_traj(results, wl, 'overlaps_fc.CD')
    xs = np.arange(arr.shape[1])
    mean, std = arr.mean(0), arr.std(0)
    ax.plot(xs, mean, label=f'warmup={wl}', color=colors[wl])
    ax.fill_between(xs, mean - std, mean + std, alpha=0.15, color=colors[wl])
ax.set_xlabel('Training step')
ax.set_ylabel('CD_fc')
ax.set_title('CD_fc over training for each warmup length')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('gap_warmup_sweep_fig1_traj.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fig 2: Final CD / margin / acc vs warmup length
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = [
    ('overlaps_fc.CD', 'CD_fc (final)', axes[0]),
    ('soft_margin_C', 'Soft margin C (final)', axes[1]),
    ('accuracy_D', 'Accuracy D (final)', axes[2]),
]
for metric_path, ylabel, ax in metrics:
    means, stds = [], []
    for wl in warmup_lengths:
        arr = get_metric_traj(results, wl, metric_path)
        # average last 10 steps
        final = arr[:, -10:].mean(axis=1)
        means.append(final.mean())
        stds.append(final.std())
    ax.errorbar(warmup_lengths, means, yerr=stds, fmt='o-', capsize=4)
    ax.set_xlabel('n_warmup_long')
    ax.set_ylabel(ylabel)
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3)
fig.suptitle('Final metrics (avg last 10 steps) vs warmup length', fontsize=12)
plt.tight_layout()
plt.savefig('gap_warmup_sweep_fig2_final_vs_warmup.png', dpi=150, bbox_inches='tight')
plt.show()